# SCARA HW 모델 예제

- 이 노트북은 `SCARA/HW/model/` 에서 실행하는 것을 전제로 한다.
- `scara.yaml`, `motors/`, `reducers/` 는 한 단계 위인 `SCARA/HW/` 에 있으며,
  `robot.py` 의 `HW_ROOT` 가 이를 자동으로 찾는다. (작업 디렉터리와 무관)
- 다른 위치에서 쓰려면 `sys.path.insert(0, r'.../SCARA/HW/model')` 후 import.


In [1]:
%pip install -r requirement_hw.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# --- my module -- #
from robot import load_param_robot
from define import DEG2RAD, RAD2DEG, DEFAULT_GRAV_ACC_VEC
from dynamics import calculate_dynamics
from traj import GenTrajCoeff_conv, TG_MODE_TRAPEZOIDAL, TG_MODE_POLY5TRAJ

# -- external package -- #
import numpy as np # --> for math
from bokeh.plotting import figure, output_notebook, output_file, show # --> for graph
from bokeh.layouts import gridplot
TOOLS = "pan,wheel_zoom,box_zoom,reset,save"

In [3]:
# -- read parameter -- #
base,links,kineType,name,axisConfig,basePose,grav,eulType,\
task_rated_speed, task_acc_time, task_dec_time = load_param_robot('scara', np.zeros(6), DEFAULT_GRAV_ACC_VEC,
                                                                                        file="scara.yaml")

UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 270: character maps to <undefined>

In [ ]:
# -- 모션 파라미터 -- #
rated_speed:list = np.array([180*DEG2RAD, 180*DEG2RAD, 180*DEG2RAD]) # joint level
taskAccTime = 0.2
Tg_step2sec = 0.002 # [sec] sim 의 update period 랑 맞추는게 맞는 듯

# -- 시작점/끝점 -- #
q_start = np.array([0,   0,  0, ]) *DEG2RAD
q_end = np.array([90, 90, 90,]) *DEG2RAD

# -- 속도 레벨 -- #
vel_rate = 100

# -- tg_mode - -# 
# tg_mode = TG_MODE_TRAPEZOIDAL
tg_mode = TG_MODE_POLY5TRAJ

# -- traj -- #
Ux = GenTrajCoeff_conv(q_start, q_end, rated_speed, taskAccTime, vel_rate,Tg_step2sec=Tg_step2sec,
                       tg_mode=TG_MODE_TRAPEZOIDAL, coord="JOINT", )



In [ ]:
Q, QD, QDD = Ux.GenSamples()
data_a = calculate_dynamics(links, Q, QD, QDD, tool=None, grav=DEFAULT_GRAV_ACC_VEC, fext=np.zeros(6))
display(data_a.df)

done


,Ave.Torque[Nm],Ave.Input Speed[RPM],Max. Input Speed[RPM],A/D Torque[Nm],Max. Moment[Nm],ReducerLife[Hour],BearingLife[Hour]
J1,16.50(29.00),2136.75(2000.00),3000.00(6000.00),28.28(51.00),29.37(74.00),484840(10000),70354519(None)
J2,8.22(29.00),2136.75(2000.00),3000.00(6000.00),13.79(51.00),9.51(74.00),3927693(10000),1216916619(None)
J3,1.54(28.00),1303.42(2440.00),1830.00(7320.00),2.73(49.00),5.62(57.00),1068510122(10000),12110807884(None)


In [ ]:
def plot_pose_traj(X, color = 'black', legend_label="src", y_range=None, p = None):
    from bokeh.models import Range1d
    
    _, dof = np.shape(X)
    if p is None:
        p = [figure(tools = TOOLS) for _ in range(6)]
    for i in range(dof):  
        p[i].line(x = np.arange(len(X)), y = X[:,i], color = color, legend_label = legend_label, line_width = 2, line_alpha = 0.5)
        if y_range is not None:
            p[i].y_range = Range1d(y_range[i][0], y_range[i][1])
    return p

def grid_plot(p, width = 500, height = 250):
    grid = gridplot([p[0:3],p[3:]], width=width, height=height)
    show(grid)

In [ ]:
# output_notebook()
output_file("link.html")

In [ ]:
# 시각화 ---------------------------------------------------
rng = [[-200,200]]*3
p = plot_pose_traj(Q*RAD2DEG,    'black', "pos", )
# grid_plot(p, 500, 150)

rng = [[-200,200]]*3
p = plot_pose_traj(QD*RAD2DEG,  'blue',  "vel", p=p)
# grid_plot(p, 500, 150)

rng = [[-1000,1000]]*3
p = plot_pose_traj(QDD*RAD2DEG,  'red',  "acc", p=p)
grid_plot(p, 500, 150)

In [ ]:
# 시각화 ---------------------------------------------------
torque = data_a.TAU.copy()
jntGearRatio = np.array([100, 100, 50.5])   # scara.yaml 기준 (원본은 61로 잘못 적혀 있었음)

for i in range(3):
    torque[:,i] /= jntGearRatio[i]

# rng = [[-2,2]]*6
p2 = plot_pose_traj(torque,  'green',  "trq",)
grid_plot(p2, 500, 150)

In [ ]:
import sys
import numpy

In [ ]:
from robot import load_param_reducer

try:
    r = load_param_reducer('PQ003-H')
    print("성공:", r.sn, r.rated_torque)
except Exception as e:
    print("실패 에러:", e)

성공: PQ003-H 29
